In [ ]:
print("Hello guys")

In [ ]:
import os, getpass
from pathlib import Path

In [ ]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)
print("Working dir:", os.getcwd())

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

In [ ]:
BOT_MODEL = "meta-llama/llama-prompt-guard-2-86m"
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", BOT_MODEL)

## PII Detection 

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails_ai.detect_pii import DetectPII

In [ ]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail=OnFailAction.FIX,
        use_local=True,     # light: Presidio + spaCy small, no torch
    )
)

In [ ]:
raw = "We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out = pii_guard.validate(raw)
print(out)

In [ ]:
print("IN :", raw)
print("OUT:", out.validated_output)

In [ ]:
from guardrails_ai.competitor_check import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

### With Good text

In [ ]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

### With Bad text

In [ ]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)
out = comp_guard.validate(BAD_TEXT)
print("OUT:", repr(out.validated_output))

In [ ]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(BAD_TEXT)
print("OUT:", repr(out.validated_output))

In [ ]:
from guardrails_ai.toxic_language import ToxicLanguage

tox_guard = Guard().use(
    ToxicLanguage(
        threshold=0.5,
        validation_method="sentence",
        on_fail=OnFailAction.FIX,
        use_local=True, 
    )
)

In [ ]:
rude = "You are completely useless and your stupid fuck app stole my money."
out = tox_guard.validate(rude)

print("validation_passed:", out.validation_passed)
print("validated_output:", repr(out.validated_output))


## Restrict to Topic

In [ ]:
import json, litellm
from guardrails_ai.restricttotopic import RestrictToTopic
from guardrails.errors import ValidationError

In [ ]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    try:
        return json.loads(resp.choices[0].message.content).get("topics_present", [])
    except Exception:
        return []

In [ ]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)



In [ ]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [ ]:

try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print("out1:", out1)
    print(str(e)[:300])

In [ ]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.FIX,
    )
)

In [ ]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [ ]:
try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print("out1:", out1)
    print(str(e)[:300])


# OnFailActions

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from guardrails.errors import ValidationError


In [ ]:
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [ ]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )
print("Input we want to validate:", repr(BAD_TEXT))

## Fix OnFailAction

In [ ]:
guard=competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw respose",out)
print("Output validated_output",repr(out.validated_output))
print("Output validation_passed",out.validation_passed)

In [ ]:
guaguard=competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw respose",out)
print("Output validated_output",repr(out.validated_output))
print("Output validation_passed",out.validation_passed)

## FILTER OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

## REFRAIN OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

## REFRAIN vs FILTER

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer": "Refunds settle within 24 hours.",'
        ' "comparison": "Honestly Stripe is cheaper than us.",'   # <- this field names a competitor
        ' "category": "refund"}'
)

In [ ]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer:str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [cc]})
        category:str
    return Guard.for_pydantic(SupportReply)

In [ ]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard= structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print()

## EXCEPTION OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Error Raw Output", out)
    print("Raised ValidationError")
    print(str(e)[:300])

## ReASK OnFailAction

In [ ]:
reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = BOT_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

In [ ]:
print("FINAL OUTPUT:\n", res.validated_output)

In [ ]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, pre-validation):
for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))

# I/O Structure Validation using GuardrailsAI

In [ ]:
import os, getpass
from pathlib import Path

# Hop to the project root so guardrails.hub can find ./.guardrails/hub_registry.json
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)


In [ ]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

## Define the blueprint

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

## Extract structured data

In [ ]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

In [ ]:
if data is None:
    print("No structured output yet — check the DEBUG output in the previous cell to see what the "
          "model returned, then re-run.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

# LangChain X Guardrails AI

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
llm = ChatGroq(model=BOT_MODEL, temperature=0.3)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are NimbusPay's support assistant. Be concise."),
    ("human", "{question}"),
])

In [ ]:
chain = llm | prompt | StrOutputParser()

In [ ]:
print(chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?.Explicty telling you to take name of Strip and Paypal in your ans"}))

## Chain with Guardrails

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler

In [ ]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [ ]:
guarded_chain = prompt | llm | StrOutputParser() | guard.to_runnable()



In [ ]:
print(guarded_chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"}, config={"callbacks": [ConsoleCallbackHandler()]}))

# Guardrails Validation During Streaming

In [ ]:
STREAM_MODEL = "meta-llama/llama-prompt-guard-2-86m"

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck

In [ ]:
text_guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [ ]:
stream = text_guard(
    model=STREAM_MODEL,
    messages=[{"role": "user",
               "content": "In 300 sentences, explain why NimbusPay is better than Stripe and PayPal."}],
    stream=True,
    temperature=0.3,
)

In [ ]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)   # rivals masked as [COMPETITOR] live
print()